### Audio Extraction

In [1]:
import subprocess

video_path = "data/video/input.mp4"
audio_path = "data/audio/audio.wav"

# -y: overwrite output file if exists, -i: input file
# -ar 16000: set audio sample to 16KHz
# -ac 1: set audio to mono not stereo
subprocess.run([
    "ffmpeg", "-y", "-i", video_path,
    "-ar", "16000", "-ac", "1",
    audio_path
], check=True)

print("done:", audio_path)

done: data/audio/audio.wav


### Transcription

In [2]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(audio_path, verbose=False)

segments = result["segments"]

for parts in segments:
    print(f"[{parts['start']:.1f} - {parts['end']:.1f}] {parts['text']}")

Detected language: English


100%|██████████| 2363/2363 [00:03<00:00, 778.15frames/s]

[0.0 - 4.5]  When we started hugging face, we joked with my co-founders Julien Thomas
[4.5 - 7.6]  that we wanted to be the first company to go public with an emoji
[7.6 - 9.4]  rather than the three-letter ticker.
[9.4 - 12.3]  And the choice is the hugging face emoji, right?
[12.3 - 13.7]  Was our favorite emoji?
[13.7 - 15.0]  So we were like, OK, let's do that.
[15.0 - 17.3]  Without maybe we would keep it for a few weeks
[17.3 - 19.8]  and then the community started to put it everywhere.
[19.8 - 21.6]  You know, like on social media.
[21.6 - 23.6]  So we were like, oh, maybe we're gonna keep it.


### Generate TTS

In [3]:
import edge_tts, asyncio, nest_asyncio
nest_asyncio.apply()            # to let async code to run smoothly 

async def generate_tts(text, out_path, voice="en-US-GuyNeural"):
    communicate = edge_tts.Communicate(text, voice)         # sets a request to convert text to speech
    await communicate.save(out_path)                        # generate audio and save to out_path

for idx, seg in enumerate(segments[:3]):
    out_path = f"tts_{idx}.mp3"
    asyncio.run(generate_tts(seg["text"], out_path))

    original_dur = seg["end"] - seg["start"]
    print(f"segment {idx}: original={original_dur:.2f}s text='{seg['text'][:40]}...'")

segment 0: original=4.50s text=' When we started hugging face, we joked ...'
segment 1: original=3.14s text=' that we wanted to be the first company ...'
segment 2: original=1.80s text=' rather than the three-letter ticker....'


### To Check audio metadata

In [4]:
from mutagen.mp3 import MP3

for idx, seg in enumerate(segments[:3]):
    original_dur = seg["end"] - seg["start"]
    generated_dur = MP3(f"tts_{idx}.mp3").info.length
    diff = generated_dur - original_dur

    print(f"segment {idx}: original={original_dur:.2f}s generated={generated_dur:.2f}s")

segment 0: original=4.50s generated=4.87s
segment 1: original=3.14s generated=4.39s
segment 2: original=1.80s generated=2.76s


Due to the differences we want to test and tune stuffs

In [5]:
async def generate_tts_rate(text, out_path, rate_percent, voice="en-US-GuyNeural"):
    rate_str = f"{rate_percent:+.0f}%"
    communicate = edge_tts.Communicate(text, voice, rate=rate_str)
    await communicate.save(out_path)

for idx, seg in enumerate(segments[:3]):
    original_dur = seg["end"] - seg["start"]
    generated_dur = MP3(f"tts_{idx}.mp3").info.length

    needed_rate = ((generated_dur / original_dur) - 1) * 100

    out_path = f"tts_{idx}_adjusted.mp3"
    asyncio.run(generate_tts_rate(seg["text"], out_path, needed_rate))

    new_dur = MP3(out_path).info.length

    print(f"segment {idx}: original={original_dur:.2f}s  needed_rate={needed_rate:+.1f}%  new_dur={new_dur:.2f}s")

segment 0: original=4.50s  needed_rate=+8.3%  new_dur=4.51s
segment 1: original=3.14s  needed_rate=+39.9%  new_dur=3.14s
segment 2: original=1.80s  needed_rate=+53.3%  new_dur=1.82s


In [6]:
MAX_RATE = 20

overflow_count = 0
drift = 0.0

for idx, seg in enumerate(segments[:3]):
    original_dur = seg["end"] - seg["start"]
    natural_dur = MP3(f"tts_{idx}.mp3").info.length

    needed_rate = ((natural_dur / original_dur) - 1) * 100
    applied_rate = max(-MAX_RATE, min(MAX_RATE, needed_rate))
    capped = applied_rate != needed_rate

    out_path = f"tts_{idx}_capped.mp3"
    asyncio.run(generate_tts_rate(seg["text"], out_path, applied_rate))
    final_dur = MP3(out_path).info.length

    segment_drift = final_dur - original_dur
    drift += segment_drift

    print(f"segment {idx}: needed={needed_rate:+.1f}%  applied={applied_rate:+.1f}%  capped={capped}  "
        f"this_drift={segment_drift:+.2f}s  cumulative_drift={drift:+.2f}s")

segment 0: needed=+8.3%  applied=+8.3%  capped=False  this_drift=+0.01s  cumulative_drift=+0.01s
segment 1: needed=+39.9%  applied=+20.0%  capped=True  this_drift=+0.53s  cumulative_drift=+0.54s
segment 2: needed=+53.3%  applied=+20.0%  capped=True  this_drift=+0.50s  cumulative_drift=+1.05s


In [7]:
len(segments)

10

In [8]:
for idx in range(len(segments) - 1):
    gap = segments[idx + 1]["start"] - segments[idx]["end"]
    print(f"Gap after segment {idx} : {gap:.2f}s")

Gap after segment 0 : 0.00s
Gap after segment 1 : 0.00s
Gap after segment 2 : 0.00s
Gap after segment 3 : 0.00s
Gap after segment 4 : 0.00s
Gap after segment 5 : 0.00s
Gap after segment 6 : 0.00s
Gap after segment 7 : 0.00s
Gap after segment 8 : 0.00s


In [9]:
import librosa      # process and analyze music
import numpy as np

# y: actual audio and sr: sample rate
y, sr = librosa.load(audio_path, sr=16000)

def is_silent_at(t, window=0.1, threshold=0.01):
    start_sample = int((t - window) * sr)
    end_sample = int((t + window) * sr)
    chunks = y[max(0, start_sample): end_sample]
    return np.abs(chunks).mean() < threshold

for idx in range(len(segments) - 1):
    boundary = segments[idx]["end"]
    silent = is_silent_at(boundary)
    print(f"Boundary after segment {idx} (t={boundary:.2f}s): silent={silent}")

Boundary after segment 0 (t=4.50s): silent=False
Boundary after segment 1 (t=7.64s): silent=False
Boundary after segment 2 (t=9.44s): silent=False
Boundary after segment 3 (t=12.34s): silent=False
Boundary after segment 4 (t=13.66s): silent=False
Boundary after segment 5 (t=15.00s): silent=False
Boundary after segment 6 (t=17.28s): silent=False
Boundary after segment 7 (t=19.80s): silent=False
Boundary after segment 8 (t=21.60s): silent=False


In [10]:
for idx in range(len(segments) - 1):
    boundary = segments[idx]["end"]
    start_sample = int((boundary - 0.1) * sr)
    end_sample = int((boundary + 0.1) * sr)
    energy = np.abs(y[max(0, start_sample):end_sample]).mean()
    print(f"boundary after segment {idx}: energy={energy:.4f}")

boundary after segment 0: energy=0.0701
boundary after segment 1: energy=0.0346
boundary after segment 2: energy=0.0177
boundary after segment 3: energy=0.0609
boundary after segment 4: energy=0.0407
boundary after segment 5: energy=0.0495
boundary after segment 6: energy=0.0529
boundary after segment 7: energy=0.0308
boundary after segment 8: energy=0.0245


In [11]:
for idx in range(3):
    y_tts, sr_tts = librosa.load(f"tts_{idx}.mp3", sr=None)
    total_dur = len(y_tts) / sr_tts

    non_silent = np.where(np.abs(y_tts) > 0.01)[0]
    speech_start = non_silent[0] / sr_tts
    speech_end = non_silent[-1] / sr_tts

    print(f"segment {idx}: total={total_dur:.2f}s  speech=[{speech_start:.2f}-{speech_end:.2f}]  "
        f"lead_in={speech_start:.2f}s  trail_out={total_dur - speech_end:.2f}s")

segment 0: total=4.87s  speech=[0.20-4.01]  lead_in=0.20s  trail_out=0.87s
segment 1: total=4.39s  speech=[0.21-3.49]  lead_in=0.21s  trail_out=0.90s
segment 2: total=2.76s  speech=[0.19-1.86]  lead_in=0.19s  trail_out=0.90s


In [12]:
def trimmed_duration(path, threshold=0.01):
    y_tts, sr_tts = librosa.load(path, sr=None)
    non_silent = np.where(np.abs(y_tts) > threshold)[0]
    if len(non_silent) == 0:
        return 0.0
    return (non_silent[-1] - non_silent[0]) / sr_tts

for idx, seg in enumerate(segments[:3]):
    original_dur = seg["end"] - seg["start"]
    trimmed_dur = trimmed_duration(f"tts_{idx}.mp3")
    gap = trimmed_dur - original_dur
    print(f"Segment {idx}: original={original_dur:.2f}s trimmed={trimmed_dur:.2f}s gap={gap:+.2f}s")

Segment 0: original=4.50s trimmed=3.81s gap=-0.69s
Segment 1: original=3.14s trimmed=3.28s gap=+0.14s
Segment 2: original=1.80s trimmed=1.66s gap=-0.14s


In [27]:
def process_segment(seg, idx, voice="en-US-GuyNeural"):
    text = seg["text"]
    original_dur = seg["end"] - seg["start"]

    # generate the natural rate
    raw_path = f"tts_{idx}_raw.mp3"
    asyncio.run(generate_tts_rate(text, raw_path, 0))

    # measure real speech duration
    trimmed_dur = trimmed_duration(raw_path)
    gap = trimmed_dur - original_dur

    # only adjust if too long: capped at 20%
    if gap > 0:
        needed_rate = (trimmed_dur / original_dur - 1) * 100
        applied_rate = min(25, needed_rate)
        final_path = f"tts_{idx}_final.mp3"
        asyncio.run(generate_tts_rate(text, final_path, applied_rate))
    else:
        final_path = raw_path

    return final_path, gap

for idx, seg in enumerate(segments[:3]):
    path, gap = process_segment(seg, idx)
    print(f"Segment {idx}: gap={gap:+.2f}s -> {path}")

Segment 0: gap=-0.69s -> tts_0_raw.mp3
Segment 1: gap=+0.14s -> tts_1_final.mp3
Segment 2: gap=-0.14s -> tts_2_raw.mp3


In [14]:
from pydub import AudioSegment          # audio editing and manipulation

video_duration = segments[-1]["end"] + 1
timeline = AudioSegment.silent(duration=int(video_duration * 1000))

cursor = 0.0                            # to track when the timeline is actually free next

for idx, seg in enumerate(segments):
    path, gap = process_segment(seg, idx)
    clip = AudioSegment.from_file(path)
    clip_dur = len(clip) / 1000.0

    placement = max(seg["start"], cursor)
    start_ms = int(placement * 1000)

    timeline = timeline.overlay(clip, position=start_ms)
    cursor = placement + clip_dur

    drift = placement - seg["start"]
    print(f"segment {idx}: wanted_start={seg['start']:.2f}  actual_start={placement:.2f}  drift={drift:+.2f}s")

timeline.export("new_audio.wav", format="wav")


segment 0: wanted_start=0.00  actual_start=0.00  drift=+0.00s
segment 1: wanted_start=4.50  actual_start=4.87  drift=+0.37s
segment 2: wanted_start=7.64  actual_start=9.10  drift=+1.46s
segment 3: wanted_start=9.44  actual_start=11.86  drift=+2.42s
segment 4: wanted_start=12.34  actual_start=15.43  drift=+3.09s
segment 5: wanted_start=13.66  actual_start=17.83  drift=+4.17s
segment 6: wanted_start=15.00  actual_start=20.47  drift=+5.47s
segment 7: wanted_start=17.28  actual_start=23.81  drift=+6.53s
segment 8: wanted_start=19.80  actual_start=27.36  drift=+7.56s
segment 9: wanted_start=21.60  actual_start=30.24  drift=+8.64s


<_io.BufferedRandom name='new_audio.wav'>

In [15]:
total_gap = sum(process_segment(seg, i)[1] for i, seg in enumerate(segments))
print(f"total original speech time: {segments[-1]['end'] - segments[0]['start']:.2f}s")
print(f"total gap (overshoot) across all segments: {total_gap:.2f}s")

total original speech time: 23.64s
total gap (overshoot) across all segments: -0.14s


In [28]:
import soundfile as sf

def trim_silence(path, threshold=0.01):
    y_tts, sr_tts = librosa.load(path, sr=None)
    non_silent = np.where(np.abs(y_tts) > threshold)[0]
    if len(non_silent) == 0:
        return path
    trimmed = y_tts[non_silent[0]:non_silent[-1]]
    trimmed_path = path.rsplit(".", 1)[0] + "_trimmed.wav"
    sf.write(trimmed_path, trimmed, sr_tts)
    return trimmed_path

def process_segment(seg, idx, voice="en-US-GuyNeural"):
    text = seg["text"]
    original_dur = seg["end"] - seg["start"]

    # generate the natural rate
    raw_path = f"tts_{idx}_raw.mp3"
    asyncio.run(generate_tts_rate(text, raw_path, 0))
    
    # measure real speech duration
    raw_trimmed = trim_silence(raw_path)
    raw_trimmed_dur = librosa.get_duration(path=raw_trimmed)
    gap = raw_trimmed_dur - original_dur

    # only adjust if too long: capped at 20%
    if gap > 0:
        needed_rate = (raw_trimmed_dur / original_dur - 1) * 100
        applied_rate = min(25, needed_rate)
        adjusted_path = f"tts_{idx}_final.mp3"
        asyncio.run(generate_tts_rate(text, adjusted_path, applied_rate))
        final_path = trim_silence(adjusted_path)
    else:
        final_path = raw_trimmed

    return final_path, gap

In [29]:
from pydub import AudioSegment          # audio editing and manipulation

video_duration = segments[-1]["end"] + 1
timeline = AudioSegment.silent(duration=int(video_duration * 1000))

cursor = 0.0                            # to track when the timeline is actually free next

for idx, seg in enumerate(segments):
    path, gap = process_segment(seg, idx)
    clip = AudioSegment.from_file(path)
    clip_dur = len(clip) / 1000.0

    placement = max(seg["start"], cursor)
    start_ms = int(placement * 1000)

    timeline = timeline.overlay(clip, position=start_ms)
    cursor = placement + clip_dur

    drift = placement - seg["start"]
    print(f"segment {idx}: wanted_start={seg['start']:.2f}  actual_start={placement:.2f}  drift={drift:+.2f}s")

timeline.export("new_audio.wav", format="wav")


segment 0: wanted_start=0.00  actual_start=0.00  drift=+0.00s
segment 1: wanted_start=4.50  actual_start=4.50  drift=+0.00s
segment 2: wanted_start=7.64  actual_start=7.65  drift=+0.01s
segment 3: wanted_start=9.44  actual_start=9.44  drift=+0.00s
segment 4: wanted_start=12.34  actual_start=12.34  drift=+0.00s
segment 5: wanted_start=13.66  actual_start=13.66  drift=+0.00s
segment 6: wanted_start=15.00  actual_start=15.31  drift=+0.31s
segment 7: wanted_start=17.28  actual_start=17.55  drift=+0.27s
segment 8: wanted_start=19.80  actual_start=20.01  drift=+0.21s
segment 9: wanted_start=21.60  actual_start=21.80  drift=+0.20s


<_io.BufferedRandom name='new_audio.wav'>

In [30]:
result = subprocess.run([
    "ffmpeg", "-y",
    "-i", "data/video/input.mp4",
    "-i", "new_audio.wav",
    "-c:v", "copy",         
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-shortest",                # ends based on wht is short audio or video
    "output.mp4"
], capture_output=True, text=True)


In [31]:
import edge_tts, asyncio
voices = asyncio.run(edge_tts.list_voices())
for v in voices:
    if "en-US" in v["ShortName"] or "en-GB" in v["ShortName"]:
        print(v["ShortName"], "-", v["Gender"])

en-US-AvaNeural - Female
en-US-AndrewNeural - Male
en-US-EmmaNeural - Female
en-US-BrianNeural - Male
en-GB-LibbyNeural - Female
en-GB-MaisieNeural - Female
en-GB-RyanNeural - Male
en-GB-SoniaNeural - Female
en-GB-ThomasNeural - Male
en-US-AnaNeural - Female
en-US-AndrewMultilingualNeural - Male
en-US-AriaNeural - Female
en-US-AvaMultilingualNeural - Female
en-US-BrianMultilingualNeural - Male
en-US-ChristopherNeural - Male
en-US-EmmaMultilingualNeural - Female
en-US-EricNeural - Male
en-US-GuyNeural - Male
en-US-JennyNeural - Female
en-US-MichelleNeural - Female
en-US-RogerNeural - Male
en-US-SteffanNeural - Male


In [32]:
sample_text = "This is a quick test of how this voice sounds."
for name in ["en-US-AndrewNeural", "en-US-AriaNeural", "en-US-JennyNeural", "en-US-EricNeural"]:
    asyncio.run(generate_tts_rate(sample_text, f"preview_{name}.mp3", 0, voice=name))